# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading, exploring, and processing the [FAIR^2 Mass Spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema provided via the following URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL for FAIR^2
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata object directly
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n\nPublished: {getattr(metadata, 'datePublished', 'n/a')}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets, their @id, and their fields.
print("Available Record Sets and Fields:")
for rs in dataset.record_sets:
    print(f"- Record Set: {rs.name} (@id: {rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) [type: {field.data_type}]")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set and field references are made exclusively by `@id`.

In [ ]:
# Extract all record sets into pandas DataFrames using their @ids.
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"Record set @ids: {record_set_ids}\n")
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set '{record_set_id}'. Columns:")
        print(df.columns.tolist())
    else:
        print(f"No records found for record set '{record_set_id}'.")

# As an example, select the main protein record set (replace with your record set @id found above)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nPreview of the first rows in record set '{main_rs_id}':")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps like filtering records based on specific criteria, normalizing numeric fields, and grouping data.

Replace `<numeric_field_id>` and `<group_field_id>` below with appropriate `@id` values from your dataset.

In [ ]:
# Replace these @ids with those present in your dataset
main_df = dataframes[main_rs_id]

# Example: Use field @id for normalized abundance (e.g., 'normalized_abundance', exact @id from overview above)
# Find a numeric field
numeric_field_id = None
for rs in dataset.record_sets:
    if rs.id == main_rs_id:
        for field in rs.fields:
            if field.data_type in ['Float', 'Integer', 'Number']:
                numeric_field_id = field.id
                print(f"Selected numeric field: {field.name} (@id: {field.id})")
                break
        break

if numeric_field_id and numeric_field_id in main_df.columns:
    threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a 'group' field (e.g., sample name or experimental condition)
    # Find potential group field
    group_field_id = None
    for field in dataset.record_sets[0].fields:
        if field.data_type == 'Text' and field.id in main_df.columns:
            group_field_id = field.id
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA in the main record set data.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the main numeric field
if numeric_field_id and numeric_field_id in main_df.columns and pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

# Plot group means if group_field_id found
if 'grouped_df' in locals() and group_field_id:
    plt.figure(figsize=(12,4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to access, examine, and perform basic processing and visualization of the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using `mlcroissant`.

- All references were made via `@id` fields as prescribed by the Croissant schema.
- For detailed analyses, users should review full schema documentation, field descriptions, and tailor the EDA accordingly.

You can now extend this notebook to perform further bioinformatics, mass spectrometry analysis, or build ML pipelines using this public FAIR^2 resource.